# 🧠 MindTrack AI — ViT Facial Emotion Training

This notebook trains a Vision Transformer (ViT-Base) on FER2013 for 7-class facial emotion recognition.

**What this produces:**
- Trained model weights (`vit_fer2013_best.pt`)
- Confusion matrix on held-out test set
- Classification report with per-class F1 scores
- Training curves (loss + validation F1)
- Complete metrics summary

**Requirements:** Google Colab free tier with T4 GPU (Runtime → Change runtime type → T4 GPU)

**Estimated training time:** ~45-60 minutes on T4 GPU for 30 epochs

## Cell 1: Setup — Run This First

This cell installs all dependencies, mounts Google Drive for saving results, and configures Kaggle credentials for dataset download.

In [ ]:
# ===== SETUP — Run this first =====

# Install dependencies
!pip install -q torch torchvision transformers scikit-learn matplotlib pandas numpy kaggle pillow

# Optional: MLflow for experiment tracking
!pip install -q mlflow

# Mount Google Drive to save results
from google.colab import drive
drive.mount('/content/drive')

# Set up output directory in Drive
import os
DRIVE_OUTPUT = '/content/drive/MyDrive/MindTrack-AI-Training'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✅ Output directory: {DRIVE_OUTPUT}')

# Set up Kaggle credentials
# Option 1: Upload your kaggle.json when prompted
# Option 2: Set environment variables directly
# Uncomment and fill in your Kaggle credentials:
# os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'
# os.environ['KAGGLE_KEY'] = 'your_kaggle_key'

# Or upload kaggle.json interactively:
try:
    from google.colab import files
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print('📁 Please upload your kaggle.json file:')
        uploaded = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(list(uploaded.values())[0])
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
        print('✅ Kaggle credentials configured')
    else:
        print('✅ Kaggle credentials already present')
except Exception as e:
    print(f'⚠️ Kaggle setup: {e}')
    print('You can manually download FER2013 and upload to /content/data/datasets/fer2013/')

# Verify GPU
import torch
print(f'\n🖥️ Device: {"GPU — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (training will be slow)"}')
print(f'📦 PyTorch: {torch.__version__}')
print('\n✅ Setup complete! Proceed to Cell 2.')

## Cell 2: Download FER2013

Downloads the FER2013 dataset from Kaggle. The dataset contains 35,887 grayscale 48x48 images across 7 emotion classes:
- angry, disgust, fear, happy, sad, surprise, neutral

**Note:** If Kaggle download fails, you can manually download from https://www.kaggle.com/datasets/msambare/fer2013 and upload the extracted folder to `/content/data/datasets/fer2013/`

In [ ]:
# ===== Download FER2013 =====

import os
DATA_DIR = '/content/data/datasets/fer2013'
os.makedirs(DATA_DIR, exist_ok=True)

# Check if already downloaded
train_dir = os.path.join(DATA_DIR, 'train')
if os.path.exists(train_dir):
    total = sum(len(files) for _, _, files in os.walk(train_dir))
    test_dir = os.path.join(DATA_DIR, 'test')
    if os.path.exists(test_dir):
        total += sum(len(files) for _, _, files in os.walk(test_dir))
    print(f'✅ FER2013 already downloaded: {total} images')
else:
    print('📥 Downloading FER2013 from Kaggle...')
    !kaggle datasets download -d msambare/fer2013 -p {DATA_DIR} --unzip
    print('✅ Download complete!')

# Show dataset structure
print('\n📁 Dataset structure:')
for split in ['train', 'test']:
    split_dir = os.path.join(DATA_DIR, split)
    if os.path.exists(split_dir):
        for emotion in sorted(os.listdir(split_dir)):
            emotion_dir = os.path.join(split_dir, emotion)
            if os.path.isdir(emotion_dir):
                count = len(os.listdir(emotion_dir))
                print(f'  {split}/{emotion}: {count} images')

## Cell 3: Run Training

This cell runs the complete training pipeline. All hyperparameters are visible and easy to change.

**Training details:**
- Model: ViT-Base-Patch16-224 (pretrained on ImageNet)
- Frozen layers: 0-7 (only fine-tuning layers 8-11 + pooler + new head)
- Optimizer: AdamW with dual learning rates
- Scheduler: Cosine Annealing
- Early stopping: patience=8 on validation macro F1
- Class imbalance: WeightedRandomSampler + weighted CrossEntropyLoss

In [ ]:
# ===== Run Training =====

import sys
sys.path.insert(0, '/content')

# Copy training script to /content if not already there
import shutil
os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/results/splits', exist_ok=True)
os.makedirs('/content/data/pretrained', exist_ok=True)

# Import training function
# If running from the repo, the script is in ml_training/
# For Colab, we import directly
from train_facial_vit import train
from pathlib import Path

# ===== HYPERPARAMETERS — Change these as needed =====
HYPERPARAMS = {
    'data_dir': Path(DATA_DIR),
    'num_epochs': 30,           # Max epochs (early stopping may stop earlier)
    'batch_size': 32,           # 32 works well on T4 GPU (16GB)
    'lr_pretrained': 2e-5,      # Learning rate for frozen-then-unfrozen ViT layers
    'lr_head': 1e-4,            # Learning rate for new classification head
    'weight_decay': 0.01,       # L2 regularization
    'patience': 8,              # Early stopping patience (epochs without improvement)
    'quick_test': None,         # Set to e.g. 100 for a quick validation run
}

print('🏋️ Starting training with these hyperparameters:')
for k, v in HYPERPARAMS.items():
    print(f'  {k}: {v}')
print()

# Run training
metrics = train(**HYPERPARAMS)

print('\n✅ Training complete! Proceed to Cell 4 to view results.')

## Cell 4: View Results

Displays the confusion matrix, classification report, and training curves generated during training.

In [ ]:
# ===== View Results =====

import json
from IPython.display import display, Image as IPImage, HTML
import matplotlib.pyplot as plt

# Display confusion matrix
print('📊 Confusion Matrix (held-out test set):')
cm_path = '/content/results/confusion_matrix.png'
if os.path.exists(cm_path):
    display(IPImage(filename=cm_path, width=600))
else:
    print('  ⚠️ Confusion matrix not found')

# Display training curves
print('\n📈 Training Curves:')
curves_path = '/content/results/training_curves.png'
if os.path.exists(curves_path):
    display(IPImage(filename=curves_path, width=800))
else:
    print('  ⚠️ Training curves not found')

# Print classification report
print('\n📋 Classification Report:')
report_path = '/content/results/classification_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
    print('-' * 55)
    for cls in ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']:
        if cls in report:
            r = report[cls]
            print(f"{cls:<12} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1-score']:>10.4f} {r.get('support', 'N/A'):>10}")
    print('-' * 55)
    if 'macro avg' in report:
        r = report['macro avg']
        print(f"{'macro avg':<12} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1-score']:>10.4f} {r.get('support', 'N/A'):>10}")
else:
    print('  ⚠️ Classification report not found')

# Print metrics summary
print('\n📊 Metrics Summary:')
metrics_path = '/content/results/metrics_summary.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        summary = json.load(f)
    print(f"  Best Val F1:      {summary.get('best_val_f1', 'N/A')}")
    print(f"  Test Macro F1:    {summary.get('test_macro_f1', 'N/A')}")
    print(f"  Test Accuracy:    {summary.get('test_accuracy', 'N/A')}")
    print(f"  Best Epoch:       {summary.get('best_epoch', 'N/A')}")
    print(f"  Early Stopped:    {summary.get('early_stopped', 'N/A')}")
    print(f"  Training Time:    {summary.get('total_training_time_minutes', 'N/A')} min")
    print(f"  CPU Latency:      {summary.get('cpu_inference_latency_ms', 'N/A')} ms")
else:
    print('  ⚠️ Metrics summary not found')

print('\n✅ Results displayed. Proceed to Cell 5 to download.')

## Cell 5: Download Results

Zips the results folder and model weights, copies to Google Drive, and provides a download link.

In [ ]:
# ===== Download Results =====

import shutil
from google.colab import files

# Copy results to Google Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/MindTrack-AI-Training'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy all results
for item in ['confusion_matrix.png', 'training_curves.png', 'classification_report.json',
             'metrics_summary.json', 'roc_auc_scores.json']:
    src = f'/content/results/{item}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_OUTPUT, item))

# Copy model weights
model_src = '/content/data/pretrained/vit_fer2013_best.pt'
if os.path.exists(model_src):
    shutil.copy2(model_src, os.path.join(DRIVE_OUTPUT, 'vit_fer2013_best.pt'))
    print(f'✅ Model weights copied to Google Drive')

# Create zip for local download
zip_path = '/content/mindtrack_training_results'
shutil.make_archive(zip_path, 'zip', '/content/results')
print(f'📦 Results zipped: {zip_path}.zip')

# Also zip model weights separately (can be large)
if os.path.exists(model_src):
    model_zip = '/content/mindtrack_model_weights'
    os.makedirs('/tmp/model_export', exist_ok=True)
    shutil.copy2(model_src, '/tmp/model_export/vit_fer2013_best.pt')
    shutil.make_archive(model_zip, 'zip', '/tmp/model_export')
    print(f'📦 Model weights zipped: {model_zip}.zip')

print(f'\n☁️ All results saved to Google Drive: {DRIVE_OUTPUT}')
print('\n📥 Click below to download to your computer:')

# Download results zip
try:
    files.download(f'{zip_path}.zip')
except Exception:
    print(f'  Download from: {zip_path}.zip')

print('\n✅ Done! Copy vit_fer2013_best.pt to your project at data/pretrained/ to enable real inference.')